# Analysis of the Berka dataset

**Goal:** get to know the 8 tables in the dataset, understand how they relate to each other, and spot quality issues before cleaning the data.

**Data:** Berka dataset (PKDD'99), real anonymized data from a Czech bank between 1993 and 1998. The files are in `data/raw/` and use `;` as separator.

**Business question:** Which patterns in customers' transactional behavior predict that they will default on their loan, and which segments concentrate the highest risk?

In [2]:
import pandas as pd
import numpy as np

## Loan

Loans table: one row per loan. It is the central table of the analysis because it contains the target variable (`status`).

Meaning of `status` according to the dataset documentation:

| status | Meaning |
|---|---|
| A | Contract finished, loan paid |
| B | Contract finished, loan **not** paid |
| C | Contract running, payments up to date |
| D | Contract running, client in debt |

The `date` column comes as a number in `YYMMDD` format (e.g. `930705` = July 5, 1993).

In [3]:
loan = pd.read_csv("../data/raw/loan.csv", sep=";")
loan.head()

,loan_id,account_id,date,amount,duration,payments,status
0,5314,1787,930705,96396,12,8033.0,B
1,5316,1801,930711,165960,36,4610.0,A
2,6863,9188,930728,127080,60,2118.0,A
3,5325,1843,930803,105804,36,2939.0,A
4,7240,11013,930906,274740,60,4579.0,A


In [4]:
print(loan.shape)
print(loan["status"].value_counts())
loan.duplicated().value_counts()   # verificar duplicados

(682, 7)
status
C    403
A    203
D     45
B     31
Name: count, dtype: int64


False    682
Name: count, dtype: int64

**Findings — Loan**

- 682 loans and 7 columns, no duplicate rows.
- Most loans are healthy: C = 403 and A = 203. The problematic loans are D = 45 and B = 31 (see the default section).

## Trans

Transactions table: every money movement of every account. It is the largest table in the dataset and the basis for measuring customers' transactional behavior.

The values are in Czech. Translation according to the documentation:

- `type`: `PRIJEM` = income (credit), `VYDAJ` = expense (debit).
- `operation`: `VKLAD` = cash deposit, `VYBER` = cash withdrawal, `VYBER KARTOU` = credit card withdrawal, `PREVOD Z UCTU` = collection from another bank, `PREVOD NA UCET` = remittance to another bank.
- `k_symbol`: `UROK` = interest credited, `SANKC. UROK` = penalty interest for negative balance, `SLUZBY` = service fee, `SIPO` = household payment, `DUCHOD` = pension, `POJISTNE` = insurance, `UVER` = loan payment.

`bank` is read as text so that bank codes are not converted to numbers.

In [5]:
trans =  pd.read_csv("../data/raw/trans.csv", sep= ";", dtype= {"bank": str})
trans.head()

,trans_id,account_id,date,type,operation,amount,balance,k_symbol,bank,account
0,695247,2378,930101,PRIJEM,VKLAD,700.0,700.0,NaN,NaN,NaN
1,171812,576,930101,PRIJEM,VKLAD,900.0,900.0,NaN,NaN,NaN
2,207264,704,930101,PRIJEM,VKLAD,1000.0,1000.0,NaN,NaN,NaN
3,1117247,3818,930101,PRIJEM,VKLAD,600.0,600.0,NaN,NaN,NaN
4,579373,1972,930102,PRIJEM,VKLAD,400.0,400.0,NaN,NaN,NaN


In [6]:
print(trans.shape)
trans.info()

(1056320, 10)
<class 'pandas.DataFrame'>
RangeIndex: 1056320 entries, 0 to 1056319
Data columns (total 10 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   trans_id    1056320 non-null  int64  
 1   account_id  1056320 non-null  int64  
 2   date        1056320 non-null  int64  
 3   type        1056320 non-null  str    
 4   operation   873206 non-null   str    
 5   amount      1056320 non-null  float64
 6   balance     1056320 non-null  float64
 7   k_symbol    574439 non-null   str    
 8   bank        273508 non-null   str    
 9   account     295389 non-null   float64
dtypes: float64(3), int64(3), str(4)
memory usage: 80.6 MB


In [7]:
print(trans["type"].value_counts(dropna=False))
print(trans["operation"].value_counts(dropna=False))

type
VYDAJ     634571
PRIJEM    405083
VYBER      16666
Name: count, dtype: int64
operation
VYBER             434918
PREVOD NA UCET    208283
NaN               183114
VKLAD             156743
PREVOD Z UCTU      65226
VYBER KARTOU        8036
Name: count, dtype: int64


In [8]:
trans[trans["operation"].isna()]

,trans_id,account_id,date,type,operation,amount,balance,k_symbol,bank,account
137,3591421,1844,930131,PRIJEM,NaN,12.6,3754.6,UROK,NaN,NaN
138,3617490,2632,930131,PRIJEM,NaN,47.5,15411.5,UROK,NaN,NaN
139,3579543,1493,930131,PRIJEM,NaN,13.3,5222.3,UROK,NaN,NaN
140,3558798,866,930131,PRIJEM,NaN,37.1,13181.1,UROK,NaN,NaN
141,3453599,6473,930131,PRIJEM,NaN,17.0,21078.0,UROK,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1056315,3626622,2906,981231,PRIJEM,NaN,62.3,13729.4,UROK,NaN,NaN
1056316,3627616,2935,981231,PRIJEM,NaN,81.3,19544.9,UROK,NaN,NaN
1056317,3625403,2869,981231,PRIJEM,NaN,60.2,14638.2,UROK,NaN,NaN
1056318,3626683,2907,981231,PRIJEM,NaN,107.5,23453.0,UROK,NaN,NaN


In [9]:
print(trans[trans["operation"].isna()]["k_symbol"].value_counts(dropna=False))
trans[trans["operation"].isna()]["type"].value_counts(dropna=False)

k_symbol
UROK    183114
Name: count, dtype: int64


type
PRIJEM    183114
Name: count, dtype: int64

**Finding — empty `operation` is not an error**

All 183,114 transactions without `operation` have `type = PRIJEM` and `k_symbol = UROK`. They are the interest the bank credits automatically: there is no client operation to record. During cleaning they can be filled with their own label (e.g. `INTEREST`) instead of being treated as missing.

In [10]:
trans["k_symbol"].value_counts(dropna=False)

k_symbol
NaN            481881
UROK           183114
SLUZBY         155832
SIPO           118065
                53433
DUCHOD          30338
POJISTNE        18500
UVER            13580
SANKC. UROK      1577
Name: count, dtype: int64

In [11]:
trans.isna().sum()   # Cuenta cuantos NaN hay por columnas

trans_id           0
account_id         0
date               0
type               0
operation     183114
amount             0
balance            0
k_symbol      481881
bank          782812
account       760931
dtype: int64

**Findings — Trans**

- 1,056,320 transactions and 10 columns.
- `type` has a third value, `VYBER` (16,666 rows), which the documentation does not mention: it only defines `PRIJEM` and `VYDAJ`. Since `VYBER` means withdrawal, these are probably expenses and should be merged with `VYDAJ`. To be confirmed during cleaning.
- `k_symbol` has two forms of missing values: 481,881 `NaN` and 53,433 blank strings (`" "`). They need to be unified.
- `bank` and `account` are empty in more than 70% of the rows. This is expected: they are only filled when the transaction involves another bank.
- `SANKC. UROK` (1,577 penalty-interest transactions for negative balance) is a direct signal of financial trouble; it could be used as a credit-risk feature.

## Client

Clients table. Sex is not a column: it is hidden in `birth_number`, which has `YYMMDD` format. According to the documentation, women have 50 added to the month (e.g. `706213` = woman born on December 13, 1970).

In [12]:
client = pd.read_csv("../data/raw/client.csv",sep=";")
print(client.shape)
client.head()

(5369, 3)


,client_id,birth_number,district_id
0,1,706213,18
1,2,450204,1
2,3,406009,1
3,4,561201,5
4,5,605703,5


In [13]:
mes = (client["birth_number"] // 100) % 100
mes.describe()


count    5369.000000
mean       31.154032
std        25.307357
min         1.000000
25%         6.000000
50%        12.000000
75%        57.000000
max        62.000000
Name: birth_number, dtype: float64

In [14]:
mes.value_counts().sort_index() # Verificamos que los datos y el mes nos da un dato interesante segun la documentacion y es que del 1 al 12 son masculinos y del 51 al 62 femenino

birth_number
1     233
2     236
3     218
4     223
5     246
6     217
7     239
8     232
9     209
10    231
11    216
12    224
51    221
52    192
53    209
54    225
55    221
56    195
57    226
58    238
59    260
60    235
61    210
62    213
Name: count, dtype: int64

In [15]:
client["sex"] = np.where(mes <13,"M","F")

In [16]:
client

,client_id,birth_number,district_id,sex
0,1,706213,18,F
1,2,450204,1,M
2,3,406009,1,F
3,4,561201,5,M
4,5,605703,5,F
...,...,...,...,...
5364,13955,456030,1,F
5365,13956,430406,1,M
5366,13968,680413,61,M
5367,13971,626019,67,F


In [17]:
client["sex"].value_counts()

sex
M    2724
F    2645
Name: count, dtype: int64

**Findings — Client**

- 5,369 clients.
- The month extracted from `birth_number` only takes values 1–12 and 51–62, which confirms the documented rule.
- With that, the `sex` column was created: 2,724 men (M) and 2,645 women (F), an almost even split.
- Pending: also decode the full birth date to compute the client's age.

## Account & disp

- `account`: one row per bank account.
- `disp` (disposition): links clients to accounts and shows each client's role on the account: `OWNER` (account holder) or `DISPONENT` (authorized user).

`frequency` indicates how often the statement is issued; `POPLATEK MESICNE` = monthly.

In [18]:
account = pd.read_csv("../data/raw/account.csv",sep=";")
disp = pd.read_csv("../data/raw/disp.csv",sep=";")

In [19]:
print(account.head())
disp.head()

   account_id  district_id         frequency    date
0         576           55  POPLATEK MESICNE  930101
1        3818           74  POPLATEK MESICNE  930101
2         704           55  POPLATEK MESICNE  930101
3        2378           16  POPLATEK MESICNE  930101
4        2632           24  POPLATEK MESICNE  930102


,disp_id,client_id,account_id,type
0,1,1,1,OWNER
1,2,2,2,OWNER
2,3,3,2,DISPONENT
3,4,4,3,OWNER
4,5,5,3,DISPONENT


In [20]:
disp["type"].value_counts()

type
OWNER        4500
DISPONENT     869
Name: count, dtype: int64

In [21]:
print(account.shape)
print(disp.shape)

(4500, 4)
(5369, 4)


**Findings — Account & disp**

- 4,500 accounts and 5,369 records in `disp`, one per client.
- There are exactly 4,500 `OWNER`s: each account has a single holder. The 869 `DISPONENT`s are additional users.
- Therefore, to link an account (and its loan) to a single person, the `OWNER` is used.

## Merge

To study who defaults, each loan has to be joined with its client: `loan → disp (on account_id) → client (on client_id)`.

An account can have a holder and an authorized user, so only `OWNER`s are kept so that each loan is linked to a single person.

In [22]:
owners = disp[disp["type"]== "OWNER"]
owners.shape

(4500, 4)

In [23]:
loan_client = pd.merge(loan,owners,on="account_id") # clientes que tienen prestamos

In [24]:
loan_client = pd.merge(loan_client,client,on="client_id")

In [26]:
prueba = pd.merge(loan, disp, on = "account_id")

In [27]:
prueba.shape

(827, 10)

In [ ]:
loan_client.shape

(682, 13)

**Finding — why filter OWNER**

- Joining `loan` with the `OWNER`s keeps all 682 loans: one loan, one client.
- Joining `loan` with all of `disp` (variable `prueba`) gives 827 rows: loans from accounts with an authorized user are duplicated, which would inflate any count or rate.

## Do men or women default more?

The target variable `is_default` is defined: it is 1 if the loan has `status` B (finished, not paid) or D (running, in debt), and 0 if it is A or C.

In [ ]:
loan_client["is_default"] = loan_client["status"].isin(["B","D"]).astype(int) # Se agrega columna para saber si esta en mora y se coloca en binario en vez de booleano

In [ ]:
loan_client["is_default"].sum()

np.int64(76)

In [ ]:
loan_client.groupby("sex")["is_default"].mean()

sex
F    0.117816
M    0.104790
Name: is_default, dtype: float64

In [ ]:
loan_client

,loan_id,account_id,date,amount,duration,payments,status,disp_id,client_id,type,birth_number,district_id,sex,is_default
0,5314,1787,930705,96396,12,8033.0,B,2166,2166,OWNER,475722,30,F,1
1,5316,1801,930711,165960,36,4610.0,A,2181,2181,OWNER,680722,46,M,0
2,6863,9188,930728,127080,60,2118.0,A,11006,11314,OWNER,360602,45,M,0
3,5325,1843,930803,105804,36,2939.0,A,2235,2235,OWNER,405420,14,F,0
4,7240,11013,930906,274740,60,4579.0,A,13231,13539,OWNER,780907,63,M,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
677,4989,105,981205,352704,48,7348.0,C,127,127,OWNER,506227,21,F,0
678,5221,1284,981205,52512,12,4376.0,C,1554,1554,OWNER,441215,55,M,0
679,6402,6922,981206,139488,24,5812.0,C,8356,8356,OWNER,690904,3,M,0
680,5346,1928,981206,55632,24,2318.0,C,2342,2342,OWNER,481218,70,M,0


In [ ]:
loan_client["status"].value_counts()

status
C    403
A    203
D     45
B     31
Name: count, dtype: int64

**Findings — default by sex**

- 76 of 682 loans are in default (B + D), an overall rate of 11.1%.
- Default rate: women 11.8%, men 10.5%. The difference is 1.3 percentage points.
- With only 682 loans, this difference is small and does not allow us to claim that sex influences default; a statistical test would be needed (e.g. chi-square).
- Caveat: `is_default` mixes finished (A/B) and running (C/D) contracts. A C loan could still go into default.

## Card

Issued cards. They are linked to `disp` through `disp_id`, not directly to the account. There are three types: `classic`, `junior` and `gold`.

In [ ]:
card = pd.read_csv("../data/raw/card.csv", sep = ";")

In [ ]:
print(card.shape)
card.head()

(892, 4)


,card_id,disp_id,type,issued
0,1005,9285,classic,931107 00:00:00
1,104,588,classic,940119 00:00:00
2,747,4915,classic,940205 00:00:00
3,70,439,classic,940208 00:00:00
4,577,3687,classic,940215 00:00:00


In [ ]:
card.info()

<class 'pandas.DataFrame'>
RangeIndex: 892 entries, 0 to 891
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   card_id  892 non-null    int64
 1   disp_id  892 non-null    int64
 2   type     892 non-null    str  
 3   issued   892 non-null    str  
dtypes: int64(2), str(2)
memory usage: 28.0 KB


In [ ]:
print(card["type"].value_counts())

print(card.isna().sum())

type
classic    659
junior     145
gold        88
Name: count, dtype: int64
card_id    0
disp_id    0
type       0
issued     0
dtype: int64


In [ ]:
card_disp = pd.merge(disp,card,on="disp_id") # unimos dos tablas para verificar si junior tiene relacion con tipos de cuentas disponent

In [ ]:
card_disp[card_disp["type_y"]=="junior"]["type_x"].value_counts() # aqui lo descartamos no tiene nada que ver.

type_x
OWNER    145
Name: count, dtype: int64

**Findings — Card**

- 892 cards, no null values: classic = 659, junior = 145, gold = 88.
- `issued` has the time `00:00:00` attached to the date; it needs to be removed and converted to a date during cleaning.
- Hypothesis rejected: `junior` cards are not tied to authorized users. All 145 belong to account holders (`OWNER`).

## Order

Permanent payment orders (automatic payments scheduled from an account). Its `k_symbol` uses almost the same codes as `trans`, plus `LEASING`.

In [ ]:
order =pd.read_csv("../data/raw/order.csv", sep=";")

In [ ]:
print(order.shape)
order.info()

(6471, 6)
<class 'pandas.DataFrame'>
RangeIndex: 6471 entries, 0 to 6470
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   order_id    6471 non-null   int64  
 1   account_id  6471 non-null   int64  
 2   bank_to     6471 non-null   str    
 3   account_to  6471 non-null   int64  
 4   amount      6471 non-null   float64
 5   k_symbol    6471 non-null   str    
dtypes: float64(1), int64(3), str(2)
memory usage: 303.5 KB


In [ ]:
order.head()

,order_id,account_id,bank_to,account_to,amount,k_symbol
0,29401,1,YZ,87144583,2452.0,SIPO
1,29402,2,ST,89597016,3372.7,UVER
2,29403,2,QR,13943797,7266.0,SIPO
3,29404,3,WX,83084338,1135.0,SIPO
4,29405,3,CD,24485939,327.0,


In [ ]:
order["k_symbol"].value_counts(dropna=False)

k_symbol
SIPO        3502
            1379
UVER         717
POJISTNE     532
LEASING      341
Name: count, dtype: int64

In [ ]:
trans["k_symbol"].value_counts(dropna=True)

k_symbol
UROK           183114
SLUZBY         155832
SIPO           118065
                53433
DUCHOD          30338
POJISTNE        18500
UVER            13580
SANKC. UROK      1577
Name: count, dtype: int64

**Findings — Order**

- 6,471 permanent orders and 6 columns, no `NaN`.
- But `k_symbol` has 1,379 blank strings: these are missing values that `isna()` does not detect.
- Compared with `trans`: `LEASING` only appears in `order`, while `UROK`, `SLUZBY`, `DUCHOD` and `SANKC. UROK` only appear in `trans`. This makes sense, because the latter are generated by the bank and are not payments scheduled by the client.
- `UVER` (loan payment) appears in both tables; it can help match clients with loans.

## District

Demographic data for the 77 districts. The columns come as `A1`–`A16`; their meaning according to the documentation:

| Column | Meaning |
|---|---|
| A1 | `district_id` |
| A2 | District name |
| A3 | Region |
| A4 | Number of inhabitants |
| A5–A8 | Municipalities by size (<500, 500–1999, 2000–9999, ≥10000 inhabitants) |
| A9 | Number of cities |
| A10 | % urban population |
| A11 | Average salary |
| A12 | Unemployment rate 1995 |
| A13 | Unemployment rate 1996 |
| A14 | Entrepreneurs per 1000 inhabitants |
| A15 | Crimes committed in 1995 |
| A16 | Crimes committed in 1996 |

In [ ]:
district = pd.read_csv("../data/raw/district.csv",sep=";")

In [ ]:
district.shape

(77, 16)

In [ ]:
district.head()

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,A16
0,1,Hl.m. Praha,Prague,1204953,0,0,0,1,1,100.0,12541,0.29,0.43,167,85677,99107
1,2,Benesov,central Bohemia,88884,80,26,6,2,5,46.7,8507,1.67,1.85,132,2159,2674
2,3,Beroun,central Bohemia,75232,55,26,4,1,5,41.7,8980,1.95,2.21,111,2824,2813
3,4,Kladno,central Bohemia,149893,63,29,6,2,6,67.4,9753,4.64,5.05,109,5244,5892
4,5,Kolin,central Bohemia,95616,65,30,4,1,6,51.4,9307,3.85,4.43,118,2616,3040


In [ ]:
district.dtypes  #Identificamos unas anomalias en A12 y A15 que deberian estar en float y int

A1       int64
A2         str
A3         str
A4       int64
A5       int64
A6       int64
A7       int64
A8       int64
A9       int64
A10    float64
A11      int64
A12        str
A13    float64
A14      int64
A15        str
A16      int64
dtype: object

In [ ]:
print(district["A15"].sort_values())  # identificamos en que filas estan las anomalias
print(district["A12"].sort_values())

16     1003
26     1029
21     1089
30     1328
62     1562
      ...  
0     85677
71     9672
69     9878
18      999
68        ?
Name: A15, Length: 77, dtype: str
0     0.29
10    0.45
9     0.56
26    0.65
34    1.02
      ... 
39    6.49
69    6.63
37    7.08
38    7.34
68       ?
Name: A12, Length: 77, dtype: str


In [ ]:
district[district["A12"]=="?"]

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,A16
68,69,Jesenik,north Moravia,42821,4,13,5,1,3,48.4,8173,?,7.01,124,?,1358


In [ ]:
district.isna().sum()

A1     0
A2     0
A3     0
A4     0
A5     0
A6     0
A7     0
A8     0
A9     0
A10    0
A11    0
A12    0
A13    0
A14    0
A15    0
A16    0
dtype: int64

**Findings — District**

- 77 districts and 16 columns, which need to be renamed with descriptive names.
- `A12` (unemployment 1995) and `A15` (crimes 1995) were read as text instead of numbers.
- The cause: district 69 (Jesenik) has `?` in those two columns.
- `isna()` returns 0 because pandas does not recognize `?` as missing. Solution: read the file with `na_values="?"` and then decide how to impute those values (e.g. with the 1996 figure: `A13` and `A16`).

## Summary of the first exploration

**Structure:** 8 tables joined by keys: `loan`, `trans` and `order` hang from `account`; `disp` links `account` with `client`; `card` hangs from `disp`; `client` and `account` are linked to `district`.

**Main findings**

1. There are 682 loans, 76 of which are in default (11.1%). That is the target variable.
2. The client's sex is encoded in `birth_number` (month + 50 for women). It has already been decoded.
3. The default difference between women (11.8%) and men (10.5%) is small and not conclusive.
4. To join loans with clients only `OWNER` must be used; otherwise loans are duplicated (682 → 827 rows).

**Quality issues for cleaning (`src/transform.py`)**

| Table | Issue | Proposed action |
|---|---|---|
| loan, trans, account, card and client (birth_number) | Dates as `YYMMDD` numbers | Convert to date |
| trans | Empty `operation` on interest (`UROK`) | Fill with its own label |
| trans | Undocumented `type = VYBER` | Merge with `VYDAJ` (to confirm) |
| trans, order | `k_symbol` with `NaN` and blank strings | Unify as missing |
| card | `issued` with time `00:00:00` | Remove the time and convert to date |
| district | Unnamed columns `A1`–`A16` | Rename |
| district | `?` in `A12` and `A15` (Jesenik) | Read with `na_values="?"` and impute |

**Next steps:** compute the client's age, build behavioral features from `trans` (average balance, income, use of `SANKC. UROK`) and bring these findings into `docs/data_dictionary.md`.